# Agentic AI Underwriting — Evaluation Notebook

This notebook compares **three underwriting pipelines** (from the paper, Section 5) against the synthetic applications generated by `bop_workbench.py`:

1. **Single-LLM baseline** — one LLM call, global + business-specific guidelines inlined.
2. **Naive RAG** — retrieval over the underwriting guidebook PDF, then one LLM call.
3. **Agentic RAG** — multi-node LangGraph pipeline with reflection, third-party checks, and a logistic risk score.

The heavy work (running 1,905 LLM evaluations) is done by `run_eval.py`. This notebook is a **viewer** for the precomputed results. Re-run `python run_eval.py --workers 5` from `BOP_Prompt_Workspace/` to refresh.

**Data-leakage safeguards:** every app is passed through `eval_pipelines.sanitize_app_for_eval()` before any pipeline sees it. The `ground_truth`, `chosen_reason`, `scenario`, and `decision` fields the generator writes are stripped so the pipelines see only the application form + third-party data.

## Setup

In [ ]:
import os, json
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

HERE      = Path('.').resolve()
WORKSPACE = HERE / 'BOP_Prompt_Workspace' if (HERE / 'BOP_Prompt_Workspace').exists() else HERE
OUTPUTS   = WORKSPACE / 'outputs'
DATA_DIR  = WORKSPACE / 'data'

# Locate the latest batch (the one produced by run_batch.py + recompute_ground_truth.py)
batches = sorted([p for p in OUTPUTS.glob('apps_*') if p.is_dir()])
assert batches, f'No batches under {OUTPUTS}. Run run_batch.py first.'
BATCH = batches[-1]
print(f'Using batch: {BATCH.name}')

EVAL_DIR     = BATCH / 'eval'
RESULTS_CSV  = EVAL_DIR / 'eval_results.csv'
SUMMARY_JSON = EVAL_DIR / "eval_summary.json"


## Input batch overview

In [ ]:
# Generation stats from the batch
with open(BATCH / 'all_apps.jsonl') as f:
    gen_rows = [json.loads(l) for l in f]
print(f'Generated apps: {len(gen_rows)}')
print('By intended scenario:')
for sc, n in Counter(r['scenario'] for r in gen_rows).most_common():
    print(f'  {sc:<26} {n}')

# Ground-truth labels from recompute_ground_truth.py
with open(BATCH / 'ground_truth.jsonl') as f:
    gt_rows = [json.loads(l) for l in f]
print()
print(f'Relabeled apps: {len(gt_rows)}')
print('Ground-truth decisions:')
for d, n in Counter(r['decision'] for r in gt_rows).most_common():
    print(f'  {d:<25} {n}')


## Pipeline definitions

The three pipelines live in `BOP_Prompt_Workspace/eval_pipelines.py`. Each takes a flat application dict + third-party dict and returns `{'decision': ACCEPT|REJECT|REFER_TO_HUMAN_REVIEW, 'reason': str, ...}`.

The sanitization function `sanitize_app_for_eval()` strips every ground-truth signal from the app JSON before any pipeline sees it.

In [ ]:
import sys
sys.path.insert(0, str(WORKSPACE))
import eval_pipelines as ep

print('Available pipelines:')
for name in ('SingleLLMPipeline', 'NaiveRAGPipeline', 'AgenticRAGPipeline'):
    print(f'  - ep.{name}')
print()
print('Sanitization removes top-level keys:', sorted(ep._LEAKAGE_TOP_KEYS))


## Top-line results

`run_eval.py` writes:
- `eval_results.csv` — one row per app, all pipeline decisions + reasons + latencies
- `eval_summary.json` — accuracy, confusion matrices, per-scenario breakdown, reason-similarity

In [ ]:
assert RESULTS_CSV.exists(), (
    f'No eval results at {RESULTS_CSV}. '
    f'Run:  cd {WORKSPACE} && python run_eval.py --workers 5'
)
df = pd.read_csv(RESULTS_CSV)
summary = json.load(open(SUMMARY_JSON))
print(f'Loaded {len(df)} eval rows.')
print(f'Pipelines evaluated: {list(summary["pipelines"].keys())}')
print()

# Accuracy table
accuracy = {
    p: {
        'overall_accuracy_%': round(100 * s['accuracy_overall'], 1),
        'n_valid':            s['n_valid'],
        'n_errors':           s['n_errors'],
        'mean_latency_sec':   s['latency_sec']['mean'],
    }
    for p, s in summary['pipelines'].items()
}


### Per-scenario accuracy

In [ ]:
sc_order = ['accept', 'reject_guideline', 'reject_logit',
            'incomplete_recoverable', 'incomplete_irrecoverable']

rows = []
for p, s in summary['pipelines'].items():
    for sc in sc_order:
        ps = s['per_scenario'].get(sc, {})
        rows.append({
            'pipeline': p, 'scenario': sc,
            'n': ps.get('n', 0),
            'accuracy_%': round(100 * ps.get('accuracy', 0.0), 1),
        })
per_sc = pd.DataFrame(rows).pivot(index='scenario', columns='pipeline', values='accuracy_%')
per_sc = per_sc.reindex(sc_order)
print('Accuracy (%) by scenario x pipeline:')


### Confusion matrices (rows = ground truth, cols = predicted)

In [ ]:
labels = ['ACCEPT', 'REJECT', 'REFER_TO_HUMAN_REVIEW']
for p, s in summary['pipelines'].items():
    cm = s['confusion_matrix_rows_gt']
    cm_df = pd.DataFrame(cm).T.reindex(index=labels, columns=labels).fillna(0).astype(int)
    print(f'\n--- {p} ---')


### Reason similarity (cosine over OpenAI embeddings, subsample)

In [ ]:
sim_rows = []
for p, s in summary['pipelines'].items():
    rs = s.get('reason_cosine_similarity', {})
    sim_rows.append({
        'pipeline': p,
        'n_sampled':   rs.get('n'),
        'mean_cosine': rs.get('mean'),
        'median_cosine': rs.get('median'),
    })


### Where do the pipelines disagree?

In [ ]:
labels = ['ACCEPT', 'REJECT', 'REFER_TO_HUMAN_REVIEW']
for p, s in summary['pipelines'].items():
    cm = s['confusion_matrix_rows_gt']
    cm_df = pd.DataFrame(cm).T.reindex(index=labels, columns=labels).fillna(0).astype(int)
    print()
    print(f'--- {p} ---')


### Single-app inspection

Browse particular apps to read each pipeline's reasoning.

In [ ]:
lat_cols = {p: df[f'{p}_latency_sec'].astype(float).dropna()
            for p in ('single_llm','naive_rag','agentic_rag')}
lat_df = pd.DataFrame({
    'mean_sec':   {p: round(s.mean(), 2) for p, s in lat_cols.items()},
    'median_sec': {p: round(s.median(), 2) for p, s in lat_cols.items()},
    'p95_sec':    {p: round(np.percentile(s, 95), 2) for p, s in lat_cols.items()},
    'total_sec':  {p: round(s.sum(), 1) for p, s in lat_cols.items()},
})
print(lat_df)
print()


### Latency profile

In [ ]:
lat_cols = {p: df[f'{p}_latency_sec'].astype(float).dropna()
            for p in ('single_llm','naive_rag','agentic_rag')}
lat_df = pd.DataFrame({
    'mean_sec':   {p: round(s.mean(), 2) for p, s in lat_cols.items()},
    'median_sec': {p: round(s.median(), 2) for p, s in lat_cols.items()},
    'p95_sec':    {p: round(np.percentile(s, 95), 2) for p, s in lat_cols.items()},
    'total_sec':  {p: round(s.sum(), 1) for p, s in lat_cols.items()},
})
print(lat_df)
